In [1]:
from model import VAE_Encoder_MNIST,Ganerator_MNIST
import torch
from torchvision import datasets, transforms
from torchvision.utils import save_image
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [2]:
import com_utils
import time
import socket
import json
import struct
from scipy.io import loadmat

In [ ]:
# 這個要做一個tcp/ip連接我的python和matlab
# 希望達到的效果:
#   1. 連接
#   2. 由python傳送bit stream 
#   3. matlab 接收後傳送ACK
#   4. python 印出收到

In [ ]:
def send_msg(sock, obj):
    payload = json.dumps(obj).encode('utf-8')
    header = struct.pack('!I', len(payload))   # 4 bytes unsigned int, big-endian
    sock.sendall(header + payload)

def recv_exact(sock, n):
    data = b''
    while len(data) < n:
        packet = sock.recv(n - len(data))
        if not packet:
            raise ConnectionError("Socket closed while receiving data")
        data += packet
    return data

def recv_msg(sock):
    header = recv_exact(sock, 4)
    msg_len = struct.unpack('!I', header)[0]
    payload = recv_exact(sock, msg_len)
    return json.loads(payload.decode('utf-8'))

In [5]:
HOST = '127.0.0.1'
PORT = 5000

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((HOST, PORT))

data_id = 0
while(1):
    # import data
    tx_data = loadmat(f'./latent_bits_M/bits_8/sample_{data_id:03d}.mat')
    bits = tx_data['bits'].flatten().astype(np.uint8)
    msg = {
        "ID": data_id,
        "type": "bits",
        "shape": list(bits.shape),
        "bits": bits.tolist()
    }

    send_msg(client, msg)
    print("Sent:", msg)

    reply = recv_msg(client)
    print("Reply from MATLAB:", reply)

    if (reply["ID"]==data_id):
        data_id += 1;
        print(data_id)

    if data_id == 199:
        end_msg = {
            "type": "end"
        }

        send_msg(client, end_msg)
        print("Sent end message")
        break;

client.close()

Sent: {'ID': 0, 'type': 'bits', 'shape': [512], 'bits': [0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="D:/Python/SemCom/saved_data",
    train=True,
    download=False,
    transform=transform
)

# dataloader
dl = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)

fixed_x, _ = next(iter(dl))
fixed_x = fixed_x[:64].to(device)

In [6]:
z_dim = 64
E = VAE_Encoder_MNIST(latent_dim=z_dim).to(device)
E.load_state_dict(torch.load("./save_model/E_invert_to_G_C_only.pth"))
E.eval()
for p in E.parameters():
    p.requires_grad = False

G = Ganerator_MNIST(z_dim=z_dim).to(device)
G.load_state_dict(torch.load("./save_model/G_wgangp_C.pth"))
G.eval()
for p in G.parameters():
    p.requires_grad = False